In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import ADASYN,SMOTE
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nevergrad as ng
from collections import Counter
from sklearn.impute import KNNImputer

In [100]:
df = pd.read_csv("./bank-additional/bank-additional/bank-additional-full.csv",delimiter=";")
# df.drop(["default","emp.var.rate","previous","pdays","cons.conf.idx"], axis=1, inplace=True)
df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

# DATA PREPROCESSING

In [101]:
# Feature engineering and dropping columns
df['duration'] = (df['duration']/60).round()

# df.drop(['day_of_week','default','pdays'], axis=1, inplace=True)
                                                          
# output to 0 & 1
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# cleaning uknown
df = df[df['job'] != 'unknown']
df = df[df['default'] != 'unknown']
df = df[df['marital'] != 'unknown']
df = df[df['education'] != 'unknown']
df = df[df['housing'] != 'unknown']
df = df[df['loan'] != 'unknown'] 
df = df[df['contact'] != 'unknown']
df = df[df['month'] != 'unknown']
df = df[df['day_of_week'] != 'unknown']

df = df.reset_index(drop=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30488 entries, 0 to 30487
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             30488 non-null  int64  
 1   job             30488 non-null  object 
 2   marital         30488 non-null  object 
 3   education       30488 non-null  object 
 4   default         30488 non-null  object 
 5   housing         30488 non-null  object 
 6   loan            30488 non-null  object 
 7   contact         30488 non-null  object 
 8   month           30488 non-null  object 
 9   day_of_week     30488 non-null  object 
 10  duration        30488 non-null  float64
 11  campaign        30488 non-null  int64  
 12  pdays           30488 non-null  int64  
 13  previous        30488 non-null  int64  
 14  poutcome        30488 non-null  object 
 15  emp.var.rate    30488 non-null  float64
 16  cons.price.idx  30488 non-null  float64
 17  cons.conf.idx   30488 non-null 

# **ENCODING**

# **OUTLIER HANDLING**


In [102]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

outlier_cols = [
    'duration', 'campaign', 'age', 'pdays', 'previous', 
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 
    'euribor3m', 'nr.employed'
]

df_num = df[outlier_cols].copy()


print("--- Starting Outlier-to-NaN Conversion (10 Columns) ---")

total_outliers = 0
for col in outlier_cols:
    Q1 = df_num[col].quantile(0.25)
    Q3 = df_num[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    if col in ['duration', 'campaign', 'pdays', 'previous', 'age']:
        lower_bound = max(0, lower_bound)
    
    outlier_mask = (df_num[col] < lower_bound) | (df_num[col] > upper_bound)
    num_outliers = outlier_mask.sum()
    df_num.loc[outlier_mask, col] = np.nan
    
    total_outliers += num_outliers
    print(f"✅ Converted {num_outliers:4d} outliers in '{col}' to NaN.")

print(f"\nTotal outliers converted across all columns: {total_outliers}")


print("\n--- Scaling and KNN Imputation ---")

scaler = StandardScaler()
imputer = KNNImputer(n_neighbors=5)
df_scaled = scaler.fit_transform(df_num)

df_imputed_scaled = imputer.fit_transform(df_scaled)
df_imputed = scaler.inverse_transform(df_imputed_scaled)
df[outlier_cols] = df_imputed

print("✅ Scaling, Imputation, and Inverse Scaling Complete.")
print("The 'df' DataFrame now has its outliers replaced by KNN-estimated values.")


--- Starting Outlier-to-NaN Conversion (10 Columns) ---
✅ Converted 2873 outliers in 'duration' to NaN.
✅ Converted 1675 outliers in 'campaign' to NaN.
✅ Converted  458 outliers in 'age' to NaN.
✅ Converted 1310 outliers in 'pdays' to NaN.
✅ Converted 4652 outliers in 'previous' to NaN.
✅ Converted    0 outliers in 'emp.var.rate' to NaN.
✅ Converted    0 outliers in 'cons.price.idx' to NaN.
✅ Converted  396 outliers in 'cons.conf.idx' to NaN.
✅ Converted    0 outliers in 'euribor3m' to NaN.
✅ Converted    0 outliers in 'nr.employed' to NaN.

Total outliers converted across all columns: 11364

--- Scaling and KNN Imputation ---
✅ Scaling, Imputation, and Inverse Scaling Complete.
The 'df' DataFrame now has its outliers replaced by KNN-estimated values.


In [103]:
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30488 entries, 0 to 30487
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             30488 non-null  float64
 1   job             30488 non-null  object 
 2   marital         30488 non-null  object 
 3   education       30488 non-null  object 
 4   default         30488 non-null  object 
 5   housing         30488 non-null  object 
 6   loan            30488 non-null  object 
 7   contact         30488 non-null  object 
 8   month           30488 non-null  object 
 9   day_of_week     30488 non-null  object 
 10  duration        30488 non-null  float64
 11  campaign        30488 non-null  float64
 12  pdays           30488 non-null  float64
 13  previous        30488 non-null  float64
 14  poutcome        30488 non-null  object 
 15  emp.var.rate    30488 non-null  float64
 16  cons.price.idx  30488 non-null  float64
 17  cons.conf.idx   30488 non-null 

# **SPLITTING**

In [104]:
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


# **ENCODING**

In [105]:
encoder_cols = ['education', 'job', 'month', 'poutcome','marital','housing','loan','contact','day_of_week','default']

target_encoder = TargetEncoder(cols=encoder_cols, smoothing=1.0) 

X_train_encoded = target_encoder.fit_transform(X_train, y_train)
X_test_encoded = target_encoder.transform(X_test)

print(f"Encoded X_train shape: {X_train_encoded.shape}")
X_trainNB, X_testNB, y_trainNB, y_testNB = X_train_encoded.copy(), X_test_encoded.copy(), y_train.copy(), y_test.copy()

print(y_train.unique())
X_train_encoded.head()

Encoded X_train shape: (21341, 20)
[1 0]


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
17256,38.0,0.114733,0.118738,0.124918,0.12657,0.122591,0.126581,0.160081,0.105177,0.127312,2.8,2.0,999.0,0.0,0.09915,-0.1,93.200,-42.0,4.153,5195.8
1446,50.0,0.119886,0.118738,0.124918,0.12657,0.122591,0.126581,0.058145,0.072113,0.127312,2.0,1.0,999.0,0.0,0.09915,1.1,93.994,-36.4,4.856,5191.0
9554,38.0,0.114733,0.118738,0.124918,0.12657,0.122591,0.126581,0.160081,0.098533,0.128799,6.0,2.8,999.0,0.0,0.09915,1.4,93.918,-42.7,4.957,5228.1
10176,26.0,0.099502,0.118738,0.122615,0.12657,0.129909,0.126471,0.160081,0.098533,0.112256,2.0,2.0,999.0,0.0,0.09915,1.4,93.918,-42.7,4.960,5228.1
236,36.0,0.119886,0.118738,0.124918,0.12657,0.122591,0.126581,0.058145,0.072113,0.127312,3.6,2.0,999.0,0.0,0.09915,1.1,93.994,-36.4,4.857,5191.0


# **ADASYNC**

In [106]:
ismote = SMOTE(random_state=42, sampling_strategy = "auto", k_neighbors = 10)
X_train_res, y_train_res = ismote.fit_resample(X_train_encoded, y_train)

X_train_res_nb, y_train_res_nb = X_train_res, y_train_res

X_train_res_nb_bot, y_train_res_nb_bot = X_train_res_nb.copy(), y_train_res_nb.copy()

print(f"\nResampled X_train shape: {X_train_res.shape}")
print(f"Class distribution after ADASYN: {Counter(y_train_res)}")
print(X_train_res_nb_bot.info())
print(y_train_res_nb_bot.info())


Resampled X_train shape: (37280, 20)
Class distribution after ADASYN: Counter({1: 18640, 0: 18640})
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37280 entries, 0 to 37279
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             37280 non-null  float64
 1   job             37280 non-null  float64
 2   marital         37280 non-null  float64
 3   education       37280 non-null  float64
 4   default         37280 non-null  float64
 5   housing         37280 non-null  float64
 6   loan            37280 non-null  float64
 7   contact         37280 non-null  float64
 8   month           37280 non-null  float64
 9   day_of_week     37280 non-null  float64
 10  duration        37280 non-null  float64
 11  campaign        37280 non-null  float64
 12  pdays           37280 non-null  float64
 13  previous        37280 non-null  float64
 14  poutcome        37280 non-null  float64
 15  emp.var.rate    3728

# **DECISION TREE**

In [107]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = DecisionTreeClassifier(random_state=42)   # reproducible tree structure
clf.fit(X_train_res, y_train_res)


y_pred = clf.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

import pandas as pd 

feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

print("\nFeature Importances:")
print(feature_importances)

Accuracy: 0.8572

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      7989
           1       0.45      0.52      0.48      1158

    accuracy                           0.86      9147
   macro avg       0.69      0.71      0.70      9147
weighted avg       0.87      0.86      0.86      9147


Confusion Matrix:
[[7238  751]
 [ 555  603]]

Feature Importances:
duration          0.427509
nr.employed       0.204894
cons.conf.idx     0.068651
age               0.045435
day_of_week       0.038370
euribor3m         0.037877
job               0.025635
education         0.024165
poutcome          0.024061
campaign          0.021013
cons.price.idx    0.018847
month             0.016245
housing           0.014143
marital           0.014044
contact           0.009267
loan              0.007222
emp.var.rate      0.002622
default           0.000000
pdays             0.000000
previous          0.000000
dtype: float64


In [108]:
# --- 6. Decision Tree Hyperparameter Tuning (using Nevergrad) ---
def evaluate_decision_tree(criterion, max_depth, min_samples_split, min_samples_leaf):
    """Evaluates a Decision Tree model using cross-validation on unresampled X_train."""

    if max_depth == -1:
        max_depth = None

    clf = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    # CV is performed on the original, encoded, but UNRESAMPLED training set
    scores = cross_val_score(clf, X_train_encoded, y_train, cv=5, scoring='accuracy') 
    return -np.mean(scores) # Nevergrad minimizes this function

param_space = ng.p.Instrumentation(
    criterion=ng.p.Choice(['gini', 'entropy', 'log_loss']),
    max_depth=ng.p.Choice([-1] + list(range(1, 21))),
    min_samples_split=ng.p.Scalar(lower=2, upper=10).set_integer_casting(),
    min_samples_leaf=ng.p.Scalar(lower=1, upper=5).set_integer_casting(),
)

optimizer = ng.optimizers.OnePlusOne(parametrization=param_space, budget=50)
best_suggestion = optimizer.minimize(
    lambda *args, **kwargs: evaluate_decision_tree(*args, **kwargs)
)

best_kwargs = best_suggestion.kwargs
best_params_dt = {
    'criterion': best_kwargs['criterion'],
    'max_depth': None if best_kwargs['max_depth'] == -1 else int(best_kwargs['max_depth']),
    'min_samples_split': int(best_kwargs['min_samples_split']),
    'min_samples_leaf': int(best_kwargs['min_samples_leaf'])
}

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

clf_best = DecisionTreeClassifier(**best_params_dt, random_state=42)
clf_best.fit(X_train_res, y_train_res)
y_pred_dt = clf_best.predict(X_test_encoded)

print("\n--- Decision Tree (Optimized) Results ---")
print(f"Best DT Params: {best_params_dt}")

# --------------------------------------------------------------------
#  TOTAL / OVERALL METRICS FOR ENTIRE DATASET
# --------------------------------------------------------------------
accuracy = accuracy_score(y_test, y_pred_dt)

precision_micro  = precision_score(y_test, y_pred_dt, average='micro', zero_division=0)
precision_macro  = precision_score(y_test, y_pred_dt, average='macro', zero_division=0)
precision_weight = precision_score(y_test, y_pred_dt, average='weighted', zero_division=0)

recall_micro  = recall_score(y_test, y_pred_dt, average='micro', zero_division=0)
recall_macro  = recall_score(y_test, y_pred_dt, average='macro', zero_division=0)
recall_weight = recall_score(y_test, y_pred_dt, average='weighted', zero_division=0)

f1_micro  = f1_score(y_test, y_pred_dt, average='micro', zero_division=0)
f1_macro  = f1_score(y_test, y_pred_dt, average='macro', zero_division=0)
f1_weight = f1_score(y_test, y_pred_dt, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")

print("\n--- TOTAL METRICS (Dataset) ---")
print(f"Precision (Micro):   {precision_micro:.4f}")
print(f"Precision (Macro):   {precision_macro:.4f}")
print(f"Precision (Weighted):{precision_weight:.4f}")

print(f"Recall (Micro):      {recall_micro:.4f}")
print(f"Recall (Macro):      {recall_macro:.4f}")
print(f"Recall (Weighted):   {recall_weight:.4f}")

print(f"F1-Score (Micro):    {f1_micro:.4f}")
print(f"F1-Score (Macro):    {f1_macro:.4f}")
print(f"F1-Score (Weighted): {f1_weight:.4f}")

# --------------------------------------------------------------------
# Per-class metrics
# --------------------------------------------------------------------
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

# Feature importance
feature_importances = pd.Series(clf_best.feature_importances_, index=X_train_encoded.columns)
print("\nTop 5 Feature Importances:")
print(feature_importances.sort_values(ascending=False))



--- Decision Tree (Optimized) Results ---
Best DT Params: {'criterion': 'entropy', 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3}
Accuracy: 0.8765

--- TOTAL METRICS (Dataset) ---
Precision (Micro):   0.8765
Precision (Macro):   0.7293
Precision (Weighted):0.8937
Recall (Micro):      0.8765
Recall (Macro):      0.7868
Recall (Weighted):   0.8765
F1-Score (Micro):    0.8765
F1-Score (Macro):    0.7525
F1-Score (Weighted): 0.8833

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.91      0.93      7989
           1       0.51      0.67      0.58      1158

    accuracy                           0.88      9147
   macro avg       0.73      0.79      0.75      9147
weighted avg       0.89      0.88      0.88      9147


Top 5 Feature Importances:
duration          0.530863
nr.employed       0.255284
cons.conf.idx     0.076469
poutcome          0.038350
cons.price.idx    0.022916
day_of_week       0.021343
month     

# **NAIVE BAYES**

In [109]:
# --- 8. Naive Bayes Hyperparameter Tuning (using Nevergrad) ---

def evaluate_nb(log10_var_smoothing):
    """Evaluates a GaussianNB model using cross-validation on unresampled X_train."""
    var_smoothing = 10.0 ** log10_var_smoothing
    clf_ = GaussianNB(var_smoothing=var_smoothing)
    # CV is performed on the original, encoded, but UNRESAMPLED training set
    scores = cross_val_score(clf_, X_trainNB, y_trainNB, cv=5, scoring='accuracy') 
    return -np.mean(scores)

param_space_nb = ng.p.Instrumentation(
    log10_var_smoothing=ng.p.Scalar(lower=-12, upper=-6)
)

optimizer_nb = ng.optimizers.OnePlusOne(parametrization=param_space_nb, budget=50)
best_suggestion_nb = optimizer_nb.minimize(lambda *a, **k: evaluate_nb(*a, **k))
best_log10_vs = float(best_suggestion_nb.kwargs['log10_var_smoothing'])
best_var_smoothing = 10.0 ** best_log10_vs

# --- 9. Final Naive Bayes Model Fitting (using Resampled Data) ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

# --- 10. Final Naive Bayes Model Fitting (using Resampled Data) ---

nb_best_ = GaussianNB(var_smoothing=best_var_smoothing)
nb_best_.fit(X_train_res_nb, y_train_res_nb)
y_pred_nb = nb_best_.predict(X_testNB)

print("\n--- Naive Bayes (Optimized) Results ---")
print(f"Best NB var_smoothing: {best_var_smoothing:.2e}")

# --------------------------------------------------------------------
#  TOTAL / OVERALL METRICS FOR ENTIRE DATASET
# --------------------------------------------------------------------
accuracy_nb = accuracy_score(y_testNB, y_pred_nb)

precision_micro_nb  = precision_score(y_testNB, y_pred_nb, average='micro', zero_division=0)
precision_macro_nb  = precision_score(y_testNB, y_pred_nb, average='macro', zero_division=0)
precision_weight_nb = precision_score(y_testNB, y_pred_nb, average='weighted', zero_division=0)

recall_micro_nb  = recall_score(y_testNB, y_pred_nb, average='micro', zero_division=0)
recall_macro_nb  = recall_score(y_testNB, y_pred_nb, average='macro', zero_division=0)
recall_weight_nb = recall_score(y_testNB, y_pred_nb, average='weighted', zero_division=0)

f1_micro_nb  = f1_score(y_testNB, y_pred_nb, average='micro', zero_division=0)
f1_macro_nb  = f1_score(y_testNB, y_pred_nb, average='macro', zero_division=0)
f1_weight_nb = f1_score(y_testNB, y_pred_nb, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy_nb:.4f}")

print("\n--- TOTAL METRICS (Dataset) ---")
print(f"Precision (Micro):    {precision_micro_nb:.4f}")
print(f"Precision (Macro):    {precision_macro_nb:.4f}")
print(f"Precision (Weighted): {precision_weight_nb:.4f}")

print(f"Recall (Micro):       {recall_micro_nb:.4f}")
print(f"Recall (Macro):       {recall_macro_nb:.4f}")
print(f"Recall (Weighted):    {recall_weight_nb:.4f}")

print(f"F1-Score (Micro):     {f1_micro_nb:.4f}")
print(f"F1-Score (Macro):     {f1_macro_nb:.4f}")
print(f"F1-Score (Weighted):  {f1_weight_nb:.4f}")

# --------------------------------------------------------------------
# Per-class metrics
# --------------------------------------------------------------------
print("\nClassification Report:")
print(classification_report(y_testNB, y_pred_nb, zero_division=0))



--- Naive Bayes (Optimized) Results ---
Best NB var_smoothing: 7.45e-07
Accuracy: 0.7276

--- TOTAL METRICS (Dataset) ---
Precision (Micro):    0.7276
Precision (Macro):    0.6143
Precision (Weighted): 0.8643
Recall (Micro):       0.7276
Recall (Macro):       0.7288
Recall (Weighted):    0.7276
F1-Score (Micro):     0.7276
F1-Score (Macro):     0.6139
F1-Score (Weighted):  0.7703

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.73      0.82      7989
           1       0.28      0.73      0.40      1158

    accuracy                           0.73      9147
   macro avg       0.61      0.73      0.61      9147
weighted avg       0.86      0.73      0.77      9147



# **T-TEST**

In [110]:
from mlxtend.evaluate import paired_ttest_kfold_cv,paired_ttest_5x2cv
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# dt = DecisionTreeClassifier(**best_params_dt, random_state=42)
# nb = GaussianNB(var_smoothing=best_var_smoothing)

t, p = paired_ttest_5x2cv(estimator1=nb_best_,
                          estimator2=clf_best,
                          X=X_train_res_nb_bot,
                          y=y_train_res_nb_bot,
                          random_seed=1)
'''
t_stat, p_value = paired_ttest_kfold_cv(
    estimator1=nb_best_,
    estimator2=clf_best,
    X=X_train_res_nb_bot,
    y=y_train_res_nb_bot,
    cv=10,
    scoring='accuracy'
)
'''
print(f"t-statistic: {t:.4f}")
print(f"p-value:     {p:.4f}")

if p < 0.05:
    print("→ Significant difference between models.")
else:
    print("→ No significant difference.")


t-statistic: -40.2022
p-value:     0.0000
→ Significant difference between models.
